# Переменные и библиотеки

In [17]:
import cv2
import numpy as np
import serial
import time

OUT_WIDTH = 1000
OUT_HEIGHT = 700
CAMERA_INDEX = 0
SCALE = 0.8

# ArUco
aruco_dict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
parameters = cv2.aruco.DetectorParameters()
detector = cv2.aruco.ArucoDetector(aruco_dict, parameters)

# Serial
COM_PORT = "COM7"
BAUD_RATE = 9600
TRIGGER_CMD = "STOP"

# Color
COLOR_DICT = {
    "BLACK":  (1, (65, 65, 65)),
    "RED":    (2, (140, 45, 40)),
    "ORANGE": (3, (205, 145, 40)),
    "YELLOW": (4, (180, 150, 70)),
    "GREEN":  (5, (35, 120, 75)),
    "BLUE":   (6, (40, 65, 110)),
    "PURPLE": (7, (110, 110, 150))
}


# Функции

In [18]:
def get_marker_ordered_crn(pts_4x2, axis_x, axis_y):
    """Упорядочивает 4 угла одного ArUco-маркера относительно осей поля"""
    center = np.mean(pts_4x2, axis=0)
    tl = tr = br = bl = center
    for pt in pts_4x2:
        dx = float(np.dot(pt - center, axis_x))
        dy = float(np.dot(pt - center, axis_y))
        if dx < 0 and dy < 0:
            tl = pt
        elif dx >= 0 and dy < 0:
            tr = pt
        elif dx >= 0 and dy >= 0:
            br = pt
        else:
            bl = pt
    return (tuple(tl), tuple(tr), tuple(br), tuple(bl))


def build_exact_homography(crn_seq):
    """Строит точную перспективную гомографию по всем 12 углам 3 маркеров"""
    it = iter(crn_seq)
    m0 = next(it)
    m1 = next(it)
    m2 = next(it)

    c0 = np.mean(m0, axis = 0)
    c1 = np.mean(m1, axis = 0)
    c2 = np.mean(m2, axis = 0)

    d01 = float(np.linalg.norm(c0 - c1))
    d12 = float(np.linalg.norm(c1 - c2))
    d02 = float(np.linalg.norm(c0 - c2))

    if d01 < 10 or d12 < 10 or d02 < 10:
        return None

    cos0 = abs(float(np.dot(c1 - c0, c2 - c0))) / (d01 * d02)
    cos1 = abs(float(np.dot(c0 - c1, c2 - c1))) / (d01 * d12)
    cos2 = abs(float(np.dot(c0 - c2, c1 - c2))) / (d02 * d12)

    if cos0 <= cos1 and cos0 <= cos2:
        m_br, c_br = m0, c0
        ma, ca, mb, cb = m1, c1, m2, c2
    elif cos1 <= cos0 and cos1 <= cos2:
        m_br, c_br = m1, c1
        ma, ca, mb, cb = m0, c0, m2, c2
    else:
        m_br, c_br = m2, c2
        ma, ca, mb, cb = m0, c0, m1, c1

    v1 = ca - c_br
    v2 = cb - c_br
    cross = float(v1[0] * v2[1] - v1[1] * v2[0])

    if cross < 0:
        m_tr, c_tr = ma, ca
        m_bl, c_bl = mb, cb
    else:
        m_tr, c_tr = mb, cb
        m_bl, c_bl = ma, ca

    axis_x = c_br - c_bl
    axis_y = c_br - c_tr

    src_bl = get_marker_ordered_crn(m_bl, axis_x, axis_y)
    src_br = get_marker_ordered_crn(m_br, axis_x, axis_y)
    src_tr = get_marker_ordered_crn(m_tr, axis_x, axis_y)

    src_pts = np.array(src_bl + src_br + src_tr, dtype = "float32")

    dst_pts = np.array((
        (54.0, 613.0), (119.0, 613.0), (119.0, 678.0), (54.0, 678.0),
        (805.0, 613.0), (870.0, 613.0), (870.0, 678.0), (805.0, 678.0),
        (897.0, 51.0), (962.0, 51.0), (962.0, 116.0), (897.0, 116.0)
    ), dtype = "float32")

    H, _ = cv2.findHomography(src_pts, dst_pts)
    return H

def analyze_cell(hsv_roi):
    """Возвращает 0 (нет шашки) или ID цвета (1..n), название и BGR-цвет"""
    h, w, _ = hsv_roi.shape
    if h == 0 or w == 0:
        return 0, None, None

    center_crop = hsv_roi[int(h * 0.3) : int(h * 0.7), int(w * 0.3) : int(w * 0.7)]
    median_hsv = np.median(center_crop.reshape(-1, 3), axis=0)

    sat = float(median_hsv[1])
    val = float(median_hsv[2])

    # Белый/серый фон имеет насыщенность < 40, а пастельный фиолетовый ~55-65
    if sat < 40 and val >= 75:
        return 0, None, None

    mean_hsv_pixel = np.uint8([[median_hsv]])
    mean_rgb = cv2.cvtColor(mean_hsv_pixel, cv2.COLOR_HSV2RGB)[0][0].astype(float)

    best_id = 0
    best_color_name = None
    best_rgb = (0, 0, 0)
    min_dist = float("inf")

    for name, (color_id, rgb) in COLOR_DICT.items():
        if name == "BLACK" and sat >= 45:
            continue
            
        dist = np.linalg.norm(mean_rgb - np.array(rgb, dtype=float))
        if dist < min_dist:
            min_dist = dist
            best_id = color_id
            best_color_name = name
            best_rgb = rgb

    if min_dist > 85:
        return 0, None, None

    bgr_color = (int(best_rgb[2]), int(best_rgb[1]), int(best_rgb[0]))
    if best_color_name == "BLACK":
        bgr_color = (50, 50, 50)

    return best_id, best_color_name, bgr_color


def send_array(ser, array):
    """Отправляет массив [i, j, color_id] на контроллер"""
    for row in array:
        line = f'{row[0]},{row[1]},{row[2]}\n'
        ser.write(line.encode('utf-8'))
        time.sleep(0.005)
    ser.write(b'E\n')  # Конец передачи


def draw_pixel_info(img, bgr_pixel, area_w=None):
    """Отрисовывает в правом верхнем углу заданной области (или всего окна) карточку"""
    b, g, r = int(bgr_pixel[0]), int(bgr_pixel[1]), int(bgr_pixel[2])
    rgb_arr = np.array([r, g, b], dtype=float)

    # Вычисляем насыщенность для корректной проверки BLACK (как в analyze_cell)
    hsv_pixel = cv2.cvtColor(np.uint8([[[b, g, r]]]), cv2.COLOR_BGR2HSV)
    sat = float(hsv_pixel[0][0][1])

    best_id = 0
    best_name = "NONE"
    best_ref_rgb = (245, 245, 245)
    min_dist = float("inf")

    for name, (color_id, ref_rgb) in COLOR_DICT.items():
        if name == "BLACK" and sat >= 45:
            continue
        dist = np.linalg.norm(rgb_arr - np.array(ref_rgb, dtype=float))
        if dist < min_dist:
            min_dist = dist
            best_id = color_id
            best_name = name
            best_ref_rgb = ref_rgb

    match_text = f"{best_name} ({best_id})"
    match_bgr = (int(best_ref_rgb[2]), int(best_ref_rgb[1]), int(best_ref_rgb[0]))

    font = cv2.FONT_HERSHEY_SIMPLEX
    scale_rgb = 0.55
    scale_sub = 0.55
    thick = 2

    # Сегменты строки RGB со своими цветами (в формате BGR для OpenCV)
    segments = [
        ("RGB: (", (0, 0, 0)),
        (f"{r}", (0, 0, 220)),
        (", ", (0, 0, 0)),
        (f"{g}", (0, 160, 0)),
        (", ", (0, 0, 0)),
        (f"{b}", (220, 0, 0)),
        (")", (0, 0, 0)),
    ]

    rgb_w = sum(cv2.getTextSize(txt, font, scale_rgb, thick)[0][0] for txt, _ in segments)
    rgb_h = cv2.getTextSize("RGB: (255)", font, scale_rgb, thick)[0][1]
    (sub_w, sub_h), _ = cv2.getTextSize(match_text, font, scale_sub, thick)

    # Размеры карточки и элементов
    box_w = 245
    box_h = 78
    sq_size = 24
    sq_gap = 10
    pad_y = 10
    row_gap = 10

    # Определяем правую границу (ширина левого окна, если передана)
    target_w = area_w if area_w is not None else img.shape[1]
    x1 = target_w - box_w - 12
    y1 = 12
    x2 = x1 + box_w
    y2 = y1 + box_h

    # Фон и рамка карточки
    cv2.rectangle(img, (x1, y1), (x2, y2), (245, 245, 245), -1)
    cv2.rectangle(img, (x1, y1), (x2, y2), (0, 0, 0), 1)

    # --- 1-я строка: RGB + квадратик пикселя ---
    row1_w = rgb_w + sq_gap + sq_size
    cur_x = x1 + (box_w - row1_w) // 2
    sq1_y = y1 + pad_y
    text1_y = sq1_y + (sq_size + rgb_h) // 2

    for txt, col in segments:
        cv2.putText(img, txt, (cur_x, text1_y), font, scale_rgb, col, thick)
        cur_x += cv2.getTextSize(txt, font, scale_rgb, thick)[0][0]

    sq1_x = cur_x + sq_gap
    cv2.rectangle(img, (sq1_x, sq1_y), (sq1_x + sq_size, sq1_y + sq_size), (b, g, r), -1)
    cv2.rectangle(img, (sq1_x, sq1_y), (sq1_x + sq_size, sq1_y + sq_size), (0, 0, 0), 1)

    # --- 2-я строка: Ближайший цвет + квадратик эталона ---
    row2_w = sub_w + sq_gap + sq_size
    sub_x = x1 + (box_w - row2_w) // 2
    sq2_y = sq1_y + sq_size + row_gap
    text2_y = sq2_y + (sq_size + sub_h) // 2

    cv2.putText(img, match_text, (sub_x, text2_y), font, scale_sub, (0, 0, 0), thick)

    sq2_x = sub_x + sub_w + sq_gap
    cv2.rectangle(img, (sq2_x, sq2_y), (sq2_x + sq_size, sq2_y + sq_size), match_bgr, -1)
    cv2.rectangle(img, (sq2_x, sq2_y), (sq2_x + sq_size, sq2_y + sq_size), (0, 0, 0), 1)


# Main

In [19]:
def main():
    ser = None
    t_serial_start = time.time()
    try:
        ser = serial.Serial(COM_PORT, BAUD_RATE, timeout=0.01)
    except serial.SerialException as e:
        print(f"UART ERROR ({COM_PORT}): {e}")
        return None, None

    cap = cv2.VideoCapture(CAMERA_INDEX, cv2.CAP_DSHOW)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 800)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 580)
    cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)

    if not cap.isOpened():
        print("Camera ERROR")
        if ser and ser.is_open:
            ser.close()
        return None, None

    elapsed = time.time() - t_serial_start
    if elapsed < 1.5:
        time.sleep(1.5 - elapsed)
    ser.reset_input_buffer()
    print(f"UART connected ({COM_PORT})")
    print("Окно открыто")

    # Переменные для отслеживания курсора в окне
    mouse_state = {"x": -1, "y": -1}

    def on_mouse(event, x, y, flags, param):
        if event == cv2.EVENT_MOUSEMOVE:
            mouse_state["x"] = x
            mouse_state["y"] = y

    cv2.namedWindow("Camera")
    cv2.setMouseCallback("Camera", on_mouse)

    # Координаты сетки
    m_left = 0.021
    m_right = 0.964
    m_top = 0.140
    m_bottom = 0.948

    ls_val_mtx = None
    saved_grid = np.zeros((6, 9), dtype=int)

    # Основной цикл
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        raw_clean = frame.copy()
        crn, ids, _ = detector.detectMarkers(frame)

        if ids is not None and len(ids) >= 3:
            uniq_marks = {}
            flat_ids = ids.flatten()
            for idx in range(len(flat_ids)):
                m_id = int(flat_ids.item(idx))
                if m_id not in uniq_marks and len(uniq_marks) < 3:
                    uniq_marks[m_id] = crn[idx].reshape(4, 2)

            if len(uniq_marks) == 3:
                H = build_exact_homography(uniq_marks.values())
                if H is not None:
                    ls_val_mtx = H

        total_count = None

        if ls_val_mtx is not None:
            warped = cv2.warpPerspective(frame, ls_val_mtx, (OUT_WIDTH, OUT_HEIGHT))
            warped = cv2.flip(warped, -1)

            hsv_warped = cv2.cvtColor(warped, cv2.COLOR_BGR2HSV)
            clean_img = warped.copy()
            display_img = warped.copy()

            x_start = int(OUT_WIDTH * m_left)
            x_end = int(OUT_WIDTH * m_right)
            y_start = int(OUT_HEIGHT * m_top)
            y_end = int(OUT_HEIGHT * m_bottom)

            cell_w = (x_end - x_start) / 10.0
            cell_h = (y_end - y_start) / 6.0

            grid = np.zeros((6, 9), dtype=int)

            for r in range(6):
                for c in range(1, 10):
                    x1 = int(x_start + c * cell_w)
                    y1 = int(y_start + r * cell_h)
                    x2 = int(x1 + cell_w)
                    y2 = int(y1 + cell_h)

                    # Рисование сетки
                    cv2.rectangle(display_img, (x1, y1), (x2, y2), (0, 0, 255), 1)

                    cell_hsv = hsv_warped[y1:y2, x1:x2]
                    color_id, clr_nm, bgr_color = analyze_cell(cell_hsv)

                    grid[r, c - 1] = color_id

                    if color_id > 0:
                        center_x = int(x1 + cell_w / 2)
                        center_y = int(y1 + cell_h / 2)
                        cv2.circle(display_img, (center_x, center_y), 16, bgr_color, -1)
                        cv2.circle(display_img, (center_x, center_y), 16, (0, 0, 0), 2)
                        cv2.putText(display_img, f"{clr_nm} ({color_id})", (x1 + 6, y2 - 8),
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2)

            saved_grid = grid.copy()
            total_count = int(np.count_nonzero(grid))
        else:
            clean_img = frame.copy()
            display_img = frame.copy()
            if ids is not None:
                cv2.aruco.drawDetectedMarkers(display_img, crn, ids)

        # Масштабируем основное окно
        small_display = cv2.resize(display_img, (0, 0), fx=SCALE, fy=SCALE)
        target_h, left_w, _ = small_display.shape

        # Подгоняем сырой кадр под ту же высоту с сохранением пропорций
        raw_h, raw_w, _ = raw_clean.shape
        raw_scale = target_h / float(raw_h)
        right_w = int(raw_w * raw_scale)
        small_raw = cv2.resize(raw_clean, (right_w, target_h))

        # Склеиваем оба изображения по горизонтали в одно
        combined_display = np.hstack((small_display, small_raw))
        # Вертикальная разделительная линия между кадрами
        cv2.line(combined_display, (left_w, 0), (left_w, target_h), (0, 0, 0), 2)

        # Компактная плашка счётчика в левом верхнем углу
        if total_count is not None:
            cnt_text = str(total_count)
            (tw, th), _ = cv2.getTextSize(cnt_text, cv2.FONT_HERSHEY_SIMPLEX, 0.85, 2)
            cv2.rectangle(combined_display, (12, 12), (62, 52), (245, 245, 245), -1)
            cv2.rectangle(combined_display, (12, 12), (62, 52), (0, 0, 0), 1)
            cv2.putText(combined_display, cnt_text, (12 + (50 - tw) // 2, 12 + (40 + th) // 2),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.85, (180, 60, 20), 2)

        # Получение цвета пикселя под курсором (учитываем, над какой половиной окна курсор)
        picked_bgr = None
        mx, my = mouse_state["x"], mouse_state["y"]

        if 0 <= my < target_h:
            if 0 <= mx < left_w:
                # Курсор над левой частью (выровненное поле)
                orig_x = int(mx / SCALE)
                orig_y = int(my / SCALE)
                h_img, w_img, _ = clean_img.shape
                if 0 <= orig_x < w_img and 0 <= orig_y < h_img:
                    picked_bgr = clean_img[orig_y, orig_x]
            elif left_w <= mx < left_w + right_w:
                # Курсор над правой частью (сырой кадр)
                orig_x = int((mx - left_w) / raw_scale)
                orig_y = int(my / raw_scale)
                if 0 <= orig_x < raw_w and 0 <= orig_y < raw_h:
                    picked_bgr = raw_clean[orig_y, orig_x]

        if picked_bgr is not None:
            draw_pixel_info(combined_display, picked_bgr, left_w)

        cv2.imshow("Camera", combined_display)

        if ser and ser.in_waiting > 0:
            raw_packet = ser.readline().decode("utf-8", errors="ignore").strip()
            if raw_packet == TRIGGER_CMD:
                break
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q') or key == ord('Q'):
            break

    cap.release()
    cv2.destroyAllWindows()
    print("Окно закрыто")

    return saved_grid, ser

In [20]:
matrix, arduino = main()

print(matrix)
if matrix is not None:    
    data = []
    for r in range(5, -1, -1):
        for c in range(8, -1, -1):
            if matrix[r][c] > 0:
                data.append([5 - r, 8 - c, int(matrix[r][c])])

    print(data)
    send_array(arduino, data)

    time.sleep(0.1)
    arduino.close()
    print(f"Данные отправлены, порт ({COM_PORT}) закрыт")
else:
    print("Поле не было обнаружено")

UART connected (COM7)
Окно открыто
Окно закрыто
[[0 0 0 0 0 0 4 0 0]
 [0 6 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0]
 [0 0 0 7 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0]
 [2 0 0 0 0 0 0 0 0]]
[[0, 8, 2], [2, 5, 7], [4, 7, 6], [5, 2, 4]]
Данные отправлены, порт (COM7) закрыт
